# 28 — Experimento A: corrida formal sobre las once series de evaluación

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md) ·
[ADR 0012](../docs/decisions/0012-adr-umbral-tau-del-experimento-a.md) ·
[ADR 0024](../docs/decisions/0024-adr-hoja-oe4-autocontenida.md) ·
[ADR 0025](../docs/decisions/0025-adr-relaciones-con-extremo-compuesto.md) ·
[notebook 23](23-jss-ensayo-general-es2015.ipynb) · issues #289 y #290 (épica #288)

**Alcance.** Registro de análisis de la corrida formal `experimento-a-formal`: procedencia,
comandos y duraciones de cada paso, cifras agregadas de extracción y de relaciones, y el puntaje
de la validación manual de la alineación. El notebook solo lee las salidas guardadas en
`reports/experiment-a/experimento-a-formal/`; no llama a ningún modelo ni a Ollama.

> **Aviso de alcance (ADR 0005).** Las series son series de **evaluación**. El notebook muestra
> solo **identificadores de serie y cifras agregadas**. No muestra ni cita frases de decisión ni
> contenido de transcripciones. No contiene ninguna cifra del Experimento B.

## Los términos, en simple

| Término | Qué significa |
|---|---|
| Serie | Un conjunto de reuniones de un mismo proyecto (por ejemplo ES2002). Las once series de evaluación quedan reservadas para el experimento. |
| Repetición | Una corrida completa de la extracción sobre una serie. Cada serie se extrae 3 veces para medir la variación entre corridas. |
| Alineación (τ) | Una decisión extraída cuenta como la de la referencia cuando su similitud supera el umbral τ; el principal es 0,70 y 0,75 y 0,80 son sensibilidades. |
| Precisión, recall, F1 | De lo extraído, qué parte está en la referencia (precisión); de la referencia, qué parte se encontró (recall); y su media armónica (F1). |
| Intervalo | Rango del 95 % calculado por remuestreo de reuniones: indica cuánto podría variar la cifra con otras reuniones. |
| Concordancia | De los pares que alguno de los dos criterios (alineación automática o persona) empareja, la fracción que ambos emparejan. |

---

# 1. La corrida y su procedencia

**Qué se hace.** Se lee el manifiesto de la corrida y se muestran los datos que permiten
reproducirla: serie de evaluación, número de repeticiones, commit, registro de congelamiento,
entorno fijado y estado del árbol de trabajo.

In [1]:
import json
import subprocess
from datetime import datetime

import pandas as pd

from afg.experiments.final_tables import a_headline_table, a_relations_table
from afg.shared.paths import EXPERIMENT_A_DIR

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

RUN = "experimento-a-formal"
RUN_DIR = EXPERIMENT_A_DIR / RUN


def read(name):
    return json.loads((RUN_DIR / name).read_text(encoding="utf-8"))


manifest = read("manifest.json")
informe = read("informe.json")
relaciones = read("relaciones.json")
validacion = read("validacion-manual.json")
pairs = pd.DataFrame(manifest["pairs"].values())

In [2]:
openkos = manifest["openkos"]
pd.Series(
    {
        "corrida": manifest["run_name"],
        "partición": manifest["partition"],
        "series": f"{len(manifest['series_ids'])} ({', '.join(manifest['series_ids'])})",
        "repeticiones por serie": manifest["runs"],
        "pares (serie, repetición) en el manifiesto": len(pairs),
        "estado del manifiesto": manifest["status"],
        "commit de la extracción": manifest["git_commit"][:12],
        "git_dirty": manifest["git_dirty"],
        "congelamiento": (
            f"ADR {manifest['freeze']['adr']}, commit {manifest['freeze']['code_commit'][:12]}"
        ),
        "huella del registro (sha256)": manifest["freeze"]["sha256"][:16] + "…",
        "entorno fijado cumplido": manifest["pinned_environment_ok"],
        "modelo generativo": (
            f"{openkos['generative_model']} "
            f"(temperatura {openkos['temperature']}, semilla {openkos['seed']})"
        ),
        "modelo de embeddings": openkos["embedding_model"],
        "OpenKOS": ", ".join(openkos["versions"]),
        "Ollama": openkos["ollama_version"],
        "informe reportable": informe["reportable"],
        "informe descriptivo": informe["descriptive"],
    },
    name="valor",
).to_frame()

,valor
corrida,experimento-a-formal
partición,evaluation
series,"11 (ES2002, ES2008, ES2014, ES2016, IS1003, IS1006, IS1008, IS1009, TS3003, TS3005, TS..."
repeticiones por serie,3
"pares (serie, repetición) en el manifiesto",33
estado del manifiesto,ok
commit de la extracción,09e46a87f82d
git_dirty,False
congelamiento,"ADR 0024, commit e346a4eec5ec"
huella del registro (sha256),d08c2453c5f68d74…


In [3]:
# El entorno fijado por versión y huella de modelo, tal como lo guarda el manifiesto.
pinned = manifest["pinned_environment"]
pd.DataFrame(
    [
        {"componente": "Ollama", "valor": pinned["ollama_version"]},
        {"componente": "OpenKOS", "valor": pinned["openkos_version"]},
    ]
    + [
        {"componente": f"huella de {m}", "valor": d[:16] + "…"}
        for m, d in pinned["model_digests"].items()
    ]
)

,componente,valor
0,Ollama,0.32.9
1,OpenKOS,0.7.0
2,huella de qwen3:8b,500a1f067a9f7826…
3,huella de gemma4:26b-a4b,001e5dafc3c77684…
4,huella de bge-m3,7907646426070047…


**Lectura.** La corrida cubre las 11 series de evaluación con 3 repeticiones (33 pares, todos `ok`). Se extrajo con el commit `09e46a87f82d`, árbol de trabajo limpio (`git_dirty = False`), bajo el congelamiento del ADR 0024 (código congelado en `e346a4eec5ec`) y con el entorno fijado cumplido (Ollama 0.32.9, OpenKOS 0.7.0 y las huellas de modelo del manifiesto). El informe quedó como reportable y no descriptivo. El registro de congelamiento posterior (ADR 0025) afecta solo a las relaciones y al informe, como se detalla en la sección 5.

---

# 2. Comandos y duración de cada paso

**Qué se hace.** Se listan los comandos corridos, en orden, con su duración. La duración de la
extracción sale del manifiesto (campo `seconds` de cada par, y la ventana entre el primer inicio
y el último término); la de las relaciones, del campo `seconds` de `relaciones.json`. Los pasos
que no guardan su duración en ninguna salida versionada se declaran como no registrados.

In [4]:
def hms(seconds):
    seconds = round(seconds)
    return f"{seconds // 3600} h {seconds % 3600 // 60:02d} min {seconds % 60:02d} s"


started = min(datetime.fromisoformat(s) for s in pairs["started_at"])
finished = max(datetime.fromisoformat(s) for s in pairs["finished_at"])
extraction_seconds = float(pairs["seconds"].sum())

# La salida de relaciones anterior al ADR 0025 está en el historial de git (merge del PR #297).
OLD_RELATIONS_COMMIT = "39c3c5e"


def previous_relations():
    try:
        out = subprocess.run(
            ["git", "show", f"{OLD_RELATIONS_COMMIT}:reports/experiment-a/{RUN}/relaciones.json"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout
        return json.loads(out)
    except subprocess.CalledProcessError, FileNotFoundError, json.JSONDecodeError:
        return None


old_relations = previous_relations()
steps = [
    {
        "paso": "Extracción (3 repeticiones por serie)",
        "comando": f"uv run afg experiment a-batch --partition evaluation --run-name {RUN}",
        "duración": hms(extraction_seconds),
        "fuente": "manifest.json (suma de `seconds` de los 33 pares)",
    },
    {
        "paso": "Relaciones, primera corrida (ADR 0024)",
        "comando": f"uv run afg experiment a-relations --c2-extraction {RUN} --run-revisions",
        "duración": hms(old_relations["seconds"]) if old_relations else "no disponible",
        "fuente": f"relaciones.json del commit {OLD_RELATIONS_COMMIT} (historial de git)",
    },
    {
        "paso": "Relaciones, rehechas con el registro del ADR 0025",
        "comando": f"uv run afg experiment a-relations --c2-extraction {RUN} --run-revisions",
        "duración": hms(relaciones["seconds"]),
        "fuente": "relaciones.json (campo `seconds`)",
    },
    {
        "paso": "Informe",
        "comando": f"uv run afg experiment a-report --c2-extraction {RUN}",
        "duración": "no registrada",
        "fuente": "ninguna salida versionada guarda su duración",
    },
    {
        "paso": "Hoja de validación manual de la alineación",
        "comando": f"uv run afg experiment a-validacion-hoja --run-name {RUN}",
        "duración": "no registrada",
        "fuente": "ninguna salida versionada guarda su duración",
    },
    {
        "paso": "Puntaje de la validación manual",
        "comando": f"uv run afg experiment a-validacion-puntaje --run-name {RUN}",
        "duración": "no registrada",
        "fuente": "ninguna salida versionada guarda su duración",
    },
]
pd.DataFrame(steps)

,paso,comando,duración,fuente
0,Extracción (3 repeticiones por serie),uv run afg experiment a-batch --partition evaluation --run-name experimento-a-formal,11 h 48 min 42 s,manifest.json (suma de `seconds` de los 33 pares)
1,"Relaciones, primera corrida (ADR 0024)",uv run afg experiment a-relations --c2-extraction experimento-a-formal --run-revisions,0 h 49 min 52 s,relaciones.json del commit 39c3c5e (historial de git)
2,"Relaciones, rehechas con el registro del ADR 0025",uv run afg experiment a-relations --c2-extraction experimento-a-formal --run-revisions,0 h 00 min 44 s,relaciones.json (campo `seconds`)
3,Informe,uv run afg experiment a-report --c2-extraction experimento-a-formal,no registrada,ninguna salida versionada guarda su duración
4,Hoja de validación manual de la alineación,uv run afg experiment a-validacion-hoja --run-name experimento-a-formal,no registrada,ninguna salida versionada guarda su duración
5,Puntaje de la validación manual,uv run afg experiment a-validacion-puntaje --run-name experimento-a-formal,no registrada,ninguna salida versionada guarda su duración


In [5]:
print("Primer inicio de extracción  :", started.isoformat(timespec="seconds"))
print("Último término de extracción :", finished.isoformat(timespec="seconds"))
print("Ventana total                :", hms((finished - started).total_seconds()))
print("Suma de los 33 pares         :", hms(extraction_seconds))
per_series = (
    pairs.groupby("series")["seconds"]
    .agg(["count", "sum", "mean", "min", "max"])
    .rename(
        columns={
            "count": "repeticiones",
            "sum": "total (s)",
            "mean": "media (s)",
            "min": "mínima (s)",
            "max": "máxima (s)",
        }
    )
    .round(0)
)
per_series["total"] = per_series["total (s)"].map(hms)
per_series

Primer inicio de extracción  : 2026-11-06T04:35:37+00:00
Último término de extracción : 2026-11-06T16:24:19+00:00
Ventana total                : 11 h 48 min 42 s
Suma de los 33 pares         : 11 h 48 min 42 s


,repeticiones,total (s),media (s),mínima (s),máxima (s),total
series,,,,,,
ES2002,3,4372.0,1457.0,1412.0,1486.0,1 h 12 min 52 s
ES2008,3,4557.0,1519.0,1511.0,1526.0,1 h 15 min 57 s
ES2014,3,3642.0,1214.0,1208.0,1224.0,1 h 00 min 42 s
ES2016,3,5264.0,1755.0,1609.0,1836.0,1 h 27 min 44 s
IS1003,3,3006.0,1002.0,978.0,1034.0,0 h 50 min 06 s
IS1006,3,2883.0,961.0,842.0,1048.0,0 h 48 min 03 s
IS1008,3,2471.0,824.0,763.0,881.0,0 h 41 min 11 s
IS1009,3,3317.0,1106.0,1064.0,1134.0,0 h 55 min 17 s
TS3003,3,3621.0,1207.0,1183.0,1246.0,1 h 00 min 21 s


**Lectura.** La extracción tomó 11 h 48 min 42 s de reloj, por debajo de las 14 h que estimaba el issue #289 (a ~26 min por par); cada par tomó entre 12 y 31 min, según la serie (ES2016 es la más lenta, IS1008 la más rápida). Las relaciones tomaron 49 min 52 s la primera vez y 44 s al rehacerlas con el registro del ADR 0025; la diferencia no se explica con la salida guardada (el campo `seconds` solo registra la duración), de modo que no se atribuye a una causa. Las duraciones del informe, de la hoja y del puntaje no están en ninguna salida versionada; los registros locales de esos pasos tampoco las guardan, por lo que se declaran como no registradas y no se estiman. Las marcas de tiempo del manifiesto están en UTC.

---

# 3. Experimento A: cifras agregadas

**Pregunta.** ¿Qué parte de las decisiones de la referencia recupera la extracción, y con qué
precisión, con la alineación τ = 0,70 (principal) y las sensibilidades 0,75 y 0,80?

**Qué se muestra.** Precision, recall y F1 con intervalos por conglomerado de reunión, agrupando
las 33 corridas (tabla `a_headline_table` del informe), después cada serie por separado y la
variación entre las 3 repeticiones. Solo agregados e identificadores de serie.

In [6]:
headline = a_headline_table(informe)
shown = headline[headline["scope"] == "origin_meetings"]
shown[
    [
        "tau",
        "reading",
        "precision",
        "precision_lower",
        "precision_upper",
        "recall",
        "recall_lower",
        "recall_upper",
        "f1",
        "f1_lower",
        "f1_upper",
        "tp",
        "fp",
        "fn",
        "n_clusters",
        "null_coverage",
    ]
].round(3)

,tau,reading,precision,precision_lower,precision_upper,recall,recall_lower,recall_upper,f1,f1_lower,f1_upper,tp,fp,fn,n_clusters,null_coverage
0,0.70,principal,0.149,0.117,0.184,0.308,0.247,0.373,0.201,0.161,0.240,257,1471,577,44,0.138
2,0.75,sensibilidad,0.100,0.073,0.129,0.206,0.152,0.265,0.134,0.100,0.169,172,1556,662,44,0.053
4,0.80,sensibilidad,0.050,0.031,0.071,0.104,0.066,0.149,0.068,0.043,0.094,87,1641,747,44,0.015


In [7]:
principal = next(e for e in informe["curve"] if e["role"] == "principal")["origin_meetings"]
rows = []
for serie, v in principal["by_series"].items():
    rows.append(
        {
            "serie": serie,
            "referencia": informe["reference_kinds"][serie],
            "precision": v["precision"],
            "recall": v["recall"],
            "f1": v["f1"],
            "f1 inf.": v["intervals"]["f1"][0],
            "f1 sup.": v["intervals"]["f1"][1],
            "tp": v["tp"],
            "fp": v["fp"],
            "fn": v["fn"],
            "reuniones": v["n_clusters"],
        }
    )
by_series = pd.DataFrame(rows).set_index("serie")
by_series.round(3)

,referencia,precision,recall,f1,f1 inf.,f1 sup.,tp,fp,fn,reuniones
serie,,,,,,,,,,
ES2002,single:gv,0.195,0.344,0.249,0.135,0.370,32,132,61,4
ES2008,adjudicated,0.180,0.250,0.210,0.122,0.259,24,109,72,4
ES2014,single:gm,0.131,0.271,0.176,0.063,0.262,26,173,70,4
ES2016,adjudicated,0.165,0.429,0.238,0.216,0.288,27,137,36,4
IS1003,adjudicated,0.098,0.227,0.137,0.000,0.202,15,138,51,4
IS1006,single:gv,0.178,0.479,0.260,0.043,0.500,23,106,25,4
IS1008,single:gm,0.079,0.364,0.130,0.047,0.198,12,139,21,4
IS1009,single:gv,0.133,0.167,0.148,0.000,0.419,12,78,60,4
TS3003,single:gv,0.178,0.355,0.237,0.142,0.412,33,152,60,4


In [8]:
pooled = principal["pooled"]
low, high = pooled["intervals"]["f1"]
print(f"F1 agrupado (τ = 0,70)        : {pooled['f1']:.3f}  IC 95 % [{low:.3f}, {high:.3f}]")
f1 = by_series["f1"]
print(
    f"F1 por serie, rango           : {f1.min():.3f} ({f1.idxmin()}) "
    f"a {f1.max():.3f} ({f1.idxmax()})"
)
print(f"F1 por serie, mediana         : {f1.median():.3f}")
across = principal["across_runs"]
print(f"Variación entre las {across['n_runs']} repeticiones (media ± desviación):")
for name in ("precision", "recall", "f1"):
    print(f"  {name:9s}: {across[name + '_mean']:.3f} ± {across[name + '_stdev']:.3f}")
pd.DataFrame(principal["per_run"])[["run", "tp", "fp", "fn", "precision", "recall", "f1"]].round(3)

F1 agrupado (τ = 0,70)        : 0.201  IC 95 % [0.161, 0.240]
F1 por serie, rango           : 0.130 (IS1008) a 0.260 (IS1006)
F1 por serie, mediana         : 0.210
Variación entre las 3 repeticiones (media ± desviación):
  precision: 0.149 ± 0.003
  recall   : 0.308 ± 0.004
  f1       : 0.201 ± 0.003


,run,tp,fp,fn,precision,recall,f1
0,0,85,492,193,0.147,0.306,0.199
1,1,85,493,193,0.147,0.306,0.199
2,2,87,486,191,0.152,0.313,0.204


In [9]:
roles = pd.DataFrame(pooled["by_role"]).T
roles["recall"] = roles["recall"].astype(float)
roles[["n_reference", "n_covered", "recall", "n_predicted", "n_correct", "precision"]].astype(
    float
).round(3)

,n_reference,n_covered,recall,n_predicted,n_correct,precision
ID,153.0,52.0,0.340,457.0,55.0,0.120
ME,147.0,40.0,0.272,368.0,45.0,0.122
PM,360.0,125.0,0.347,659.0,130.0,0.197
UI,117.0,15.0,0.128,244.0,27.0,0.111
unknown,57.0,25.0,0.439,0.0,0.0,NaN


**Lectura.** Con τ = 0,70 el F1 agrupado es 0,201 (IC 95 % de 0,161 a 0,240), con precisión de 0,149 y recall de 0,308: se recuperan 257 de las 834 decisiones de la referencia y se extraen 1.471 que no corresponden a ninguna. El valor supera el piso nulo (cobertura nula de 0,138 con τ = 0,70), y las sensibilidades bajan el F1 a 0,134 con τ = 0,75 y a 0,068 con τ = 0,80. Por serie, el F1 va de 0,130 (IS1008) a 0,260 (IS1006), con mediana de 0,210; los intervalos por serie son muy anchos porque cada una tiene solo 4 reuniones, así que las diferencias entre series son descriptivas. La variación entre las 3 repeticiones es pequeña (F1 de 0,201 ± 0,003): la medida es estable frente a la aleatoriedad de la extracción. Por rol, el recall más bajo es el de UI (0,128) y el más alto el de PM (0,347); los 57 casos de rol «unknown» no tienen precisión porque la máquina no predice ese rol.

## 3.1 Relaciones (`relaciones.json`)

Las relaciones usan la misma alineación (τ = 0,70, reuniones de origen). Una relación de la
referencia solo se encuentra si sus dos decisiones se alinean, por lo que la vista «extremo a
extremo» incluye la pérdida de decisiones; la vista «condicionado» la quita, pero tiene pocos
enlaces. Se muestran los totales de las 33 corridas.

In [10]:
rel = relaciones["pooled"]
print(
    f"Pares de oro: {rel['n_gold_pairs']}; enlaces de oro: {rel['n_gold_links']}, "
    f"alineados en ambos extremos: {rel['n_gold_links_aligned']}"
)
print(
    f"Relaciones de la máquina: {rel['n_machine_relations']}; enlaces: {rel['n_machine_links']}, "
    f"alineados: {rel['n_machine_links_aligned']}; errores de lectura: {relaciones['n_errors']}"
)
a_relations_table(informe).round(3)

Pares de oro: 243; enlaces de oro: 111, alineados en ambos extremos: 24
Relaciones de la máquina: 903; enlaces: 546, alineados: 27; errores de lectura: 0


,metric,view,successes,trials,rate,lower,upper
0,existence_end_to_end,precision,11,546,0.020,0.011,0.036
1,existence_end_to_end,recall,11,111,0.099,0.056,0.169
2,existence_conditional,precision,11,27,0.407,0.245,0.593
3,existence_conditional,recall,11,24,0.458,0.279,0.649
4,type_principal,end_to_end,6,111,0.054,0.025,0.113
5,type_principal,conditional,6,24,0.250,0.120,0.449
6,type_principal,among_detected,6,11,0.545,0.280,0.787
7,type_strict,end_to_end,3,111,0.027,0.009,0.076
8,type_strict,conditional,3,24,0.125,0.043,0.310
9,type_strict,among_detected,3,11,0.273,0.097,0.566


In [11]:
fp = rel["existence_end_to_end"]["fp_breakdown"]
pd.Series(fp, name="falsos positivos de existencia (extremo a extremo)").to_frame()

,falsos positivos de existencia (extremo a extremo)
gold_no_link,0
not_candidate,16
unaligned,519


**Lectura.** De los 111 enlaces de la referencia, 24 tienen alineados sus dos extremos y se encuentran 11, por lo que la existencia de relaciones es de 0,099 de recall (IC de 0,056 a 0,169) y de 0,020 de precisión extremo a extremo; 519 de los 535 falsos positivos corresponden a enlaces con un extremo sin alinear. Condicionando a que los dos extremos estén alineados, el recall sube a 0,458 (11 de 24) y la precisión a 0,407 (11 de 27), con intervalos amplios por el tamaño pequeño. Entre los 11 enlaces detectados, la dirección es correcta en los 11 y el tipo (vista principal) en 6. El cuello de botella está en la alineación de las decisiones y no en la clasificación de la relación una vez alineadas.

---

# 4. Validación manual de la alineación (issue #290)

**Pregunta.** ¿Coincide la alineación automática con τ = 0,70 con lo que una persona considera
la misma decisión?

**Qué se hace.** Una persona ciega a la clave respondió, para 100 pares de las once series
(repetición 0, semilla 42), si la frase extraída y la de la referencia son la misma decisión. La
mitad de los pares los empareja la alineación automática y la otra mitad son pares de la misma
reunión que descarta. Aquí se lee `validacion-manual.json`. El puntaje del código no define un
umbral de aprobación; el criterio de ≥ 0,80 viene del issue #290 y se aplica aquí.

In [12]:
# Criterio del issue #290. El puntaje (manual_validation.score_validation) no define umbral.
THRESHOLD = 0.80

v = validacion
conf = v["confusion"]
cuadro = pd.DataFrame(
    {
        "manual: misma decisión": [conf["auto_si"]["manual_si"], conf["auto_no"]["manual_si"]],
        "manual: distinta": [conf["auto_si"]["manual_no"], conf["auto_no"]["manual_no"]],
    },
    index=["automática: empareja", "automática: descarta"],
)
display(cuadro)
meets = v["concordance"] >= THRESHOLD
src = v["source"]
conf_ok, conf_missed = v["automatic_confirmed"], v["automatic_missed"]
print(
    f"Pares valorados                      : {v['n']} (series: {len(src['series'])}, "
    f"repetición {src['run']}, τ = {v['tau']})"
)
print(
    f"Pares candidatos del conjunto        : {src['n_candidate_pairs']} "
    f"(emparejados por la alineación: {src['n_automatic_matches']})"
)
print(f"Concordancia (Jaccard)               : {v['concordance']:.3f}")
print(
    f"Acuerdo observado                    : {v['n_agree']}/{v['n']} = "
    f"{v['observed_agreement']:.3f}"
)
print(f"Kappa de Cohen                       : {v['kappa']:.3f}")
print(
    f"Emparejados por la alineación        : "
    f"{conf_ok['n_manual_yes']}/{conf_ok['n']} confirmados a mano"
)
print(
    f"Descartados por la alineación        : "
    f"{conf_missed['n_manual_yes']}/{conf_missed['n']} que la persona sí empareja"
)
print(f"Criterio concordancia ≥ {THRESHOLD:.2f}          : {'CUMPLE' if meets else 'NO CUMPLE'}")

,manual: misma decisión,manual: distinta
automática: empareja,47,3
automática: descarta,7,43


Pares valorados                      : 100 (series: 11, repetición 0, τ = 0.7)
Pares candidatos del conjunto        : 1805 (emparejados por la alineación: 85)
Concordancia (Jaccard)               : 0.825
Acuerdo observado                    : 90/100 = 0.900
Kappa de Cohen                       : 0.800
Emparejados por la alineación        : 47/50 confirmados a mano
Descartados por la alineación        : 7/50 que la persona sí empareja
Criterio concordancia ≥ 0.80          : CUMPLE


**Lectura.** La concordancia es 0,825 y **cumple** el criterio de ≥ 0,80 del issue #290 (calculado arriba, no fijado a mano). El acuerdo observado es de 90 de 100 pares (0,900, kappa de 0,800). Los 10 desacuerdos son 3 pares que la alineación empareja y la persona considera distintos, y 7 pares que la alineación descarta y la persona sí empareja: la alineación se equivoca más por omisión que por exceso. Por eso las cifras con τ = 0,70 se leen como un piso algo conservador. La muestra es estratificada (50 emparejados y 50 descartados): las proporciones de cada grupo describen ese grupo, no el conjunto de las series. Como el puntaje no define umbral, el criterio ≥ 0,80 es el del issue y no una regla del código.

---

# 5. Nota sobre el ADR 0025: relaciones rehechas

**Qué pasó.** Después de la corrida se encontró que nueve relaciones de oro (11 extremos) de
ES2008, ES2014, ES2016 e IS1008 apuntaban a una decisión `compuesta`, que no existe en la
referencia de decisiones. El ADR 0025 reasignó esos extremos a una decisión hija (una relación,
un enlace; detalle en el [notebook 27](27-jss-relaciones-con-extremo-compuesto.ipynb)) y
reemplazó el registro de congelamiento. La extracción no se repitió: `src/`, `pyproject.toml`,
`uv.lock` y `.python-version` no cambiaron. Se rehicieron `a-relations` y `a-report`; la salida
anterior de `relaciones.json` está en el historial de git (PR #297).

**Qué se comprueba aquí.** Se compara la salida de relaciones anterior con la vigente y las
procedencias de ambas.

In [13]:
if old_relations is None:
    print("La salida anterior no está disponible en este repositorio (historial de git).")
else:
    new_prov = {
        "commit": relaciones["git_commit"][:12],
        "congelamiento": relaciones["freeze"]["adr"],
        "segundos": relaciones["seconds"],
    }
    old_prov = {
        "commit": old_relations["git_commit"][:12],
        "congelamiento": old_relations["freeze"]["adr"],
        "segundos": old_relations["seconds"],
    }
    display(pd.DataFrame({"anterior (PR #297)": old_prov, "vigente (ADR 0025)": new_prov}))
    print(
        "Entradas por serie y repetición idénticas :",
        old_relations["entries"] == relaciones["entries"],
    )
    print(
        "Totales agrupados idénticos               :",
        old_relations["pooled"] == relaciones["pooled"],
    )
    print(
        "Enlaces de oro / alineados (antes)        :",
        old_relations["pooled"]["n_gold_links"],
        "/",
        old_relations["pooled"]["n_gold_links_aligned"],
    )
    print(
        "Enlaces de oro / alineados (ahora)        :",
        relaciones["pooled"]["n_gold_links"],
        "/",
        relaciones["pooled"]["n_gold_links_aligned"],
    )

,anterior (PR #297),vigente (ADR 0025)
commit,09e46a87f82d,e99adfde105f
congelamiento,0024,0025
segundos,2992.078,43.59


Entradas por serie y repetición idénticas : True
Totales agrupados idénticos               : True
Enlaces de oro / alineados (antes)        : 111 / 24
Enlaces de oro / alineados (ahora)        : 111 / 24


**Lectura.** La salida de relaciones anterior al ADR 0025 y la vigente son **idénticas** en las 33 entradas por serie y repetición y en los totales agrupados (111 enlaces de oro, 24 alineados en ambos extremos, 11 detectados). Solo difieren la procedencia (commit `09e46a87f82d` frente a `e99adfde105f`; registro ADR 0024 frente a 0025) y la duración. Las cifras agregadas no cambiaron. **La causa de que no cambiaran no se verifica en este notebook:** explicarlo exigiría saber si los extremos hijos reasignados se alinean con τ = 0,70, y esa alineación necesita calcular similitudes con el modelo de embeddings, que este notebook no llama. Queda como un hecho observado (agregados idénticos), sin atribuirlo a un mecanismo.

---

# 6. Reinicios o interrupciones

**Qué se hace.** El manifiesto no guarda un campo de reanudación; la evidencia es la continuidad
de las marcas de tiempo: si la extracción se hubiera interrumpido y reanudado, entre el término
de un par y el inicio del siguiente habría una pausa. Se mide la pausa mayor.

In [14]:
ordered = pairs.assign(
    inicio=pairs["started_at"].map(datetime.fromisoformat),
    fin=pairs["finished_at"].map(datetime.fromisoformat),
).sort_values("inicio")
gaps = (ordered["inicio"].iloc[1:].to_numpy() - ordered["fin"].iloc[:-1].to_numpy()) / pd.Timedelta(
    seconds=1
)
print(
    "Pares:",
    len(ordered),
    "| estados:",
    sorted(set(pairs["status"])),
    "| con error:",
    int(pairs["error"].notna().sum()),
)
print(f"Pausa mayor entre el término de un par y el inicio del siguiente: {gaps.max():.3f} s")
print("Versiones de OpenKOS vistas:", sorted(set(pairs["openkos_version"])))
print("Decisiones sin evidencia (n_unmapped):", int(pairs["n_unmapped"].sum()))
print(
    "Citas no encontradas:",
    int(pairs["quotes_not_found"].sum()),
    "de",
    int(pairs["quotes_total"].sum()),
)

Pares: 33 | estados: ['ok'] | con error: 0
Pausa mayor entre el término de un par y el inicio del siguiente: 0.001 s
Versiones de OpenKOS vistas: ['0.7.0']
Decisiones sin evidencia (n_unmapped): 0
Citas no encontradas: 0 de 1728


**Lectura.** No hubo reinicios ni interrupciones declaradas: los 33 pares se completaron en una sola ventana continua (la pausa mayor entre un par y el siguiente es de milisegundos, igual al reloj de la ventana total y a la suma de duraciones), con una sola versión de OpenKOS, sin errores, sin decisiones sin evidencia y con las 1.728 citas encontradas. La ausencia de un campo de reanudación en el manifiesto hace que esta conclusión se apoye en la continuidad de las marcas de tiempo.

---

# 7. Límites

- Una sola corrida formal (33 pares) con un generador local pequeño (`qwen3:8b`); las cifras son
  las del experimento, no se repiten ni se ajustan.
- El intervalo es por conglomerado de reunión; con 11 series, las cifras por serie son
  descriptivas.
- La validación manual la hizo una sola persona; su concordancia describe el τ = 0,70 sobre una
  muestra estratificada (mitad emparejados, mitad descartados), no sobre todas las series.
- Este notebook no mide nada del Experimento B ni interpreta hipótesis; solo registra cifras
  guardadas.